# Shirley Show - separacion de voces en maxima calidad

Corre un **ensemble de modelos Roformer** en la GPU de Colab y entrega un MP3 320k
listo para subir a la app.

**Plan:** con Google AI Pro tienes 200 compute units/mes y acceso ampliado a
aceleradores (T4, L4, A100, G4). La **G4 (96 GB)** es la que permite el ensemble
completo con overlap alto. No incluye ejecucion en segundo plano: deja la pestana
abierta mientras corre.

**Tiempo esperado:** minutos. Una cancion de 4 min con 4 modelos suele quedar
entre 5 y 20 minutos segun GPU.


In [ ]:
# 1. Comprobar GPU e instalar. La primera vez tarda un par de minutos.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "audio-separator[cpu]"
import audio_separator
print("audio-separator listo")


In [ ]:
# 2. Subir el audio (o montar Drive si lo prefieres).
#    Alternativa Drive: from google.colab import drive; drive.mount('/content/drive')
from google.colab import files
subido = files.upload()
AUDIO = "/content/" + next(iter(subido))
print("Audio:", AUDIO)


In [ ]:
# 3. Configuracion.
#    MODE: "instrumental" (quita toda la voz) o "chorus" (quita voz lider, deja coros)
MODE = "instrumental"

#    OVERLAP: ventanas superpuestas (2-50). Mas = menos costuras, mas lento.
#    Con G4/A100 se puede subir a 12-16. Con T4 conviene 8.
OVERLAP = 8

#    SEGMENT: None usa el del modelo (recomendado). Subirlo solo si sabes que el
#    modelo fue entrenado para ello; en G4 hay margen para probar 512.
SEGMENT = None


In [ ]:
import subprocess, sys
from pathlib import Path

MODELS = {
    "instrumental": [
        "model_bs_roformer_ep_317_sdr_12.9755.ckpt",   # 17.0 dB instrumental
        "model_bs_roformer_ep_368_sdr_12.9628.ckpt",   # 17.0 dB instrumental
        "melband_roformer_inst_v1e.ckpt",              # 15.735
        "mel_band_roformer_instrumental_becruily.ckpt",# 15.342
    ],
    "chorus": [
        "bs_roformer_karaoke_frazer_becruily.ckpt",
        "bs_roformer_karaoke_anvuew.ckpt",
        "mel_band_roformer_karaoke_gabox_v2.ckpt",
        "mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt",
    ],
}

def separar(modelo, audio, outdir):
    outdir = Path(outdir); outdir.mkdir(parents=True, exist_ok=True)
    cmd = ["audio-separator", audio,
           "--model_filename", modelo,
           "--output_dir", str(outdir),
           "--output_format", "WAV",
           "--mdxc_overlap", str(OVERLAP),
           "--log_level", "WARNING"]
    if SEGMENT:
        cmd += ["--mdxc_segment_size", str(SEGMENT), "--mdxc_override_model_segment_size"]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    wavs = sorted(outdir.glob("*.wav"))
    no_vocal = [w for w in wavs if "vocal" not in w.name.lower()]
    if len(no_vocal) != 1:
        sys.exit(f"{modelo}: stems inesperados {[w.name for w in wavs]}")
    return no_vocal[0]

def promediar(stems, salida):
    entradas = []
    for s in stems:
        entradas += ["-i", str(s)]
    filt = f"amix=inputs={len(stems)}:duration=first:dropout_transition=0"
    subprocess.run(["ffmpeg","-y","-v","error",*entradas,
                    "-filter_complex",filt,"-map_metadata","-1",
                    "-codec:a","libmp3lame","-b:a","320k", salida], check=True)


In [ ]:
# 4. Ejecutar el ensemble.
modelos = MODELS[MODE]
print(f"Modo {MODE}: {len(modelos)} modelos, overlap {OVERLAP}")

stems = []
for i, m in enumerate(modelos, 1):
    print(f"=== [{i}/{len(modelos)}] {m} ===", flush=True)
    stems.append(separar(m, AUDIO, f"/content/stem_{i}"))

SALIDA = "/content/output.mp3"
promediar(stems, SALIDA)
print("Listo:", SALIDA)
!ffprobe -v error -show_entries stream=codec_name,bit_rate,duration -of default=noprint_wrappers=1 {SALIDA}


In [ ]:
# 5. Descargar el MP3.
from google.colab import files
files.download("/content/output.mp3")


## Notas

- **La calidad la da el ensemble y el overlap, no el modelo de GPU.** Los pesos son
  los mismos que correrias en cualquier tarjeta; la VRAM solo permite mas modelos
  y mas ventanas superpuestas.
- **Si un modelo falla** (descarga, OOM), baja OVERLAP antes que quitar modelos.
- **Para la app:** el MP3 se sube con la UI de pistas
  (POST /api/presentations/{id}/tracks, hasta 101 MB).
- **Sin navegador abierto:** Google AI Pro no incluye background execution. Para
  eso usa integrations/kaggle/ o deja la pestana abierta mientras corre.
